## Scraping 10-Ks and 10-Qs from EDGAR

##### Note: We have already scraped S&P 500 SEC filings. For testing purpose we are scraping the SEC filings for just 2 tickers.

## 1. Import the required modules

In [1]:
import sys
sys.version

'3.6.8 |Anaconda, Inc.| (default, Dec 29 2018, 19:04:46) \n[GCC 4.2.1 Compatible Clang 4.0.1 (tags/RELEASE_401/final)]'

In [2]:
import re
import os
import unicodedata
import pandas as pd
import numpy as np
import pickle
import requests
import bs4 as bs
from lxml import html
from tqdm import tqdm
import time

## 2. Data Scraping from SEC EDGAR 

#### Get the list of S&P500 tickers.

In [3]:
resp = requests.get('http://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
soup = bs.BeautifulSoup(resp.text, 'lxml')
table = soup.find('table', {'class': 'wikitable sortable'})
tickers = []
for row in table.findAll('tr')[1:]:
    ticker = row.findAll('td')[1].text
    tickers.append(ticker)
tickers = tickers[:2]

#### Each company has an index in SEC called CIK. Map ticker to CIK to get the corresponding filings on EDGAR.

In [4]:
url = 'http://www.sec.gov/cgi-bin/browse-edgar?CIK={}&Find=Search&owner=exclude&action=getcompany'
cik_re = re.compile(r'.*CIK=(\d{10}).*')
all_cik = {}
for ticker in tqdm(tickers):
    res = cik_re.findall(requests.get(url.format(ticker)).text)
    if len(res):
        all_cik[str(ticker).lower()] = str(res[0])
        
ciks_tick_df = pd.DataFrame.from_dict(data=all_cik, orient='index')
ciks_tick_df.reset_index(inplace=True)
ciks_tick_df.columns = ['ticker', 'cik']

ciks_tick_df.to_csv('AllSecTickers.csv', sep=',', encoding='utf-8', index=False)

100%|██████████| 2/2 [00:02<00:00,  1.16it/s]


#### Functions to save and close log files

In [5]:
#maintaining the logs 
def SaveLogFile(log_file_name, text):
    
    with open(log_file_name, "a") as log_file:
        log_file.write(text)

    return
#closing log file
def CloseLogFile(log_file_name):
    
    with open(log_file_name, 'a') as log_file:
        log_file.close()

    return

#### Path to scraped sec 10-K and 10-Q files

In [6]:
scraped10k_path = "/Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k"
scraped10q_path = "/Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q"

#### Defining function for scraping 10K files

In [7]:
def Extract10K(browse_url_base, filing_url_base, doc_url_base, cik, log_file_name):
    
    requestcount=0
    if not os.path.exists(cik):
        os.makedirs(cik)
    
    os.chdir(cik)
    print('Scraping CIK ', cik)
    
    SaveLogFile(log_file_name, 'Scraping CIK: '+cik)
            
    res = requests.get(browse_url_base % cik)
    requestcount+=1
    if(requestcount==10):
        time.sleep(1)
        requestcount=0

    if res.status_code != 200:
        text = "\nFailed at step 1.\n Failed to hit browse base URL for CIK with error " + str(res.status_code) + \
               "\nFailed URL: " + (browse_url_base % cik) + '\n'
        
        SaveLogFile(log_file_name, text)
            
        return

    SaveLogFile(log_file_name,"\nSuccessfully reached browse base UrL\n")
        
    #HTML parsing using BeautifulSoup
    soup = bs.BeautifulSoup(res.text, "lxml")

 
    html_tables = soup.find_all('table')
    
    if len(html_tables)<3:
        os.chdir('..')
        return
    

    sec_filing_tbl = pd.read_html(str(html_tables[2]), header=0)[0]
    sec_filing_tbl['Filings'] = [str(x) for x in sec_filing_tbl['Filings']]

    sec_filing_tbl = sec_filing_tbl[(sec_filing_tbl['Filings'] == '10-K') | (sec_filing_tbl['Filings'] == '10-K405')]
    sec_filing_tbl = sec_filing_tbl[(sec_filing_tbl['Filing Date'] >= '2008-01-01')]

    if len(sec_filing_tbl)==0:
        os.chdir('..')
        return
    
    sec_filing_tbl['Acc_No'] = [x.replace('\xa0',' ')
                               .split('Acc-no: ')[1]
                               .split(' ')[0] for x in sec_filing_tbl['Description']]

   
    SaveLogFile(log_file_name, "Getting all documents for this CIK\n")
    
    total_files_to_scrape = len(sec_filing_tbl)
    files_scraped = 0
    
    for index, row in sec_filing_tbl.iterrows():
        
       
        acc_no = str(row['Acc_No'])
        
        #check if file is already scraped 
        date = str(row['Filing Date'])
        if (os.path.exists(cik + '_' + date + '.html')) or os.path.exists(cik + '_' + date + '.txt'):
            SaveLogFile(log_file_name, "The file for date: " + date + " exists, acc no.: " + acc_no + "\n")
            files_scraped+=1
            continue
            
   
        docs_page = requests.get(filing_url_base % (cik, acc_no))
        requestcount+=1
        if(requestcount==10):
            time.sleep(1)
            requestcount=0
        
        
        #In case of request failure,log it and jump to the next filing 
        if docs_page.status_code != 200:
            text = "Request failed with error code " + str(docs_page.status_code) + \
                   "\nFailed URL: " + (filing_url_base % (cik, acc_no)) + '\n'
            SaveLogFile(log_file_name, text)
            continue
            
        SaveLogFile(log_file_name, "Got acc no. " + acc_no + "\n")
       
        docs_page_soup = bs.BeautifulSoup(docs_page.text, 'lxml')
        docs_html_tables = docs_page_soup.find_all('table')
        if len(docs_html_tables)==0:
            continue
        docs_table = pd.read_html(str(docs_html_tables[0]), header=0)[0]
        docs_table['Type'] = [str(x) for x in docs_table['Type']]
        
 
        docs_table = docs_table[(docs_table['Type'] == '10-K') | (docs_table['Type'] == '10-K405')]
        
        if len(docs_table)==0:
            continue
        elif len(docs_table)>0:
            docs_table = docs_table.iloc[0]
        
        docname = docs_table['Document']
        
         # check for nan values, if present then log the failure
        if str(docname) == 'nan':
            text = 'File with CIK: %s and Acc_No: %s is unavailable' % (cik, acc_no) + '\n'
            SaveLogFile(log_file_name, text)
            continue       
        
        file = requests.get(doc_url_base % (cik, acc_no.replace('-', ''), docname.replace(' iXBRL','')))
        requestcount+=1
        if(requestcount==10):
            time.sleep(1)
            requestcount=0
            
       
        if file.status_code != 200:
            os.chdir('..')
            text = "Request failed with error code " + str(file.status_code) + \
                   "\nFailed URL: " + (doc_url_base % (cik, acc_no.replace('-', ''), docname)) + '\n'
            SaveLogFile(log_file_name, text)
            os.chdir(cik)
            continue
        
        SaveLogFile(log_file_name, "Got 10K for acc no. " + acc_no + "\n")
        files_scraped+=1
        
        #save HTML or Text file with .txt extension
        if '.txt' in docname:
            filename = cik + '_' + date + '.txt'
            html_file = open(filename, 'a')
            html_file.write(file.text)
            html_file.close()
        else:
            filename = cik + '_' + date + '.html'
            html_file = open(filename, 'a')
            html_file.write(file.text)
            html_file.close()
    
    if(total_files_to_scrape!=files_scraped):
        print("Some files failed to scrape\n")
        text="Some files failed to scrape\n" + str(total_files_to_scrape) + "!="+ str(files_scraped) + "\n"
        SaveLogFile(log_file_name, text)
        
    SaveLogFile(log_file_name, "saving log files=================\n")
    CloseLogFile(log_file_name)
    os.chdir('..')
        
    return

#### define the function for scraping 10-Q filings

In [8]:
def Extract10Q(browse_url_base, filing_url_base, doc_url_base, cik, log_file_name):
    
    requestcount=0

    if not os.path.exists(cik):
        os.makedirs(cik)
    
    os.chdir(cik)
    print('Scraping CIK ', cik)
    
    SaveLogFile(log_file_name, 'Scraping CIK: '+cik)
            
    res = requests.get(browse_url_base % cik)
    requestcount+=1
    if(requestcount==10):
        time.sleep(1)
        requestcount=0
    
     #In case of request failure,log it and jump to the next filing 
    if res.status_code != 200:
        text = "\nFailed at step 1.\n Failed to hit browse base URL for CIK with error " + str(res.status_code) + \
               "\nFailed URL: " + (browse_url_base % cik) + '\n'
        
        SaveLogFile(log_file_name, text)
            
        return

   
    SaveLogFile(log_file_name,"\nSuccessfully reached browse base UrL\n")
        
    soup = bs.BeautifulSoup(res.text, "lxml")

    # fetching all the html tables
    html_tables = soup.find_all('table')
    
    
    if len(html_tables)<3:
        os.chdir('..')
        return
    
    # Parsing the html filing table
    sec_filing_tbl = pd.read_html(str(html_tables[2]), header=0)[0]
    sec_filing_tbl['Filings'] = [str(x) for x in sec_filing_tbl['Filings']]


    sec_filing_tbl = sec_filing_tbl[sec_filing_tbl['Filings'] == '10-Q']
    sec_filing_tbl = sec_filing_tbl[(sec_filing_tbl['Filing Date'] >= '2008-01-01')]
    
    if len(sec_filing_tbl)==0:
        os.chdir('..')
        return
    
  
    sec_filing_tbl['Acc_No'] = [x.replace('\xa0',' ')
                               .split('Acc-no: ')[1]
                               .split(' ')[0] for x in sec_filing_tbl['Description']]

   ..
    SaveLogFile(log_file_name, "Getting all documents for this CIK\n")
    
    total_files_to_scrape = len(sec_filing_tbl)
    files_scraped = 0
    
    for index, row in sec_filing_tbl.iterrows():
        
     
        acc_no = str(row['Acc_No'])
        
        # check if file already exist
        date = str(row['Filing Date'])
        if (os.path.exists(cik + '_' + date + '.html')) or os.path.exists(cik + '_' + date + '.txt'):
            SaveLogFile(log_file_name, "The file for date: " + date + " exists, acc no.: " + acc_no + "\n")
            files_scraped+=1
            continue
            
      
        docs_page = requests.get(filing_url_base % (cik, acc_no))
        requestcount+=1
        if(requestcount==10):
            time.sleep(1)
            requestcount=0
        
        #In case of request failure,log it and jump to the next filing 
        if docs_page.status_code != 200:
            text = "Request failed with error code " + str(docs_page.status_code) + \
                   "\nFailed URL: " + (filing_url_base % (cik, acc_no)) + '\n'
            SaveLogFile(log_file_name, text)
            continue
            
        SaveLogFile(log_file_name, "Got acc no. " + acc_no + "\n")

        docs_page_soup = bs.BeautifulSoup(docs_page.text, 'lxml')
        docs_html_tables = docs_page_soup.find_all('table')
        
        if len(docs_html_tables)==0:
            continue
            
        docs_table = pd.read_html(str(docs_html_tables[0]), header=0)[0]
        docs_table['Type'] = [str(x) for x in docs_table['Type']]
        
        docs_table = docs_table[docs_table['Type'] == '10-Q']
        
        if len(docs_table)==0:
            continue
            
        elif len(docs_table)>0:
            docs_table = docs_table.iloc[0]
        
        docname = docs_table['Document']
    
        # check for nan values, if present then log the failure
        if str(docname) == 'nan':
            text = 'File with CIK: %s and Acc_No: %s is unavailable' % (cik, acc_no) + '\n'
            SaveLogFile(log_file_name, text)
            continue       
        
       
        file = requests.get(doc_url_base % (cik, acc_no.replace('-', ''), docname.replace(' iXBRL','')))
        requestcount+=1
        if(requestcount==10):
            time.sleep(1)
            requestcount=0
            
        #In case of request failure,log it and exit
        if file.status_code != 200:
            os.chdir('..')
            text = "Request failed with error code " + str(file.status_code) + \
                   "\nFailed URL: " + (doc_url_base % (cik, acc_no.replace('-', ''), docname)) + '\n'
            SaveLogFile(log_file_name, text)
            os.chdir(cik)
            continue
        
      
        SaveLogFile(log_file_name, "Got 10Q for acc no. " + acc_no + "\n")
        files_scraped+=1
        
         #save HTML or Text file with .txt extension
        if '.txt' in docname:
            filename = cik + '_' + date + '.txt'
            html_file = open(filename, 'a')
            html_file.write(file.text)
            html_file.close()
        else:
            filename = cik + '_' + date + '.html'
            html_file = open(filename, 'a')
            html_file.write(file.text)
            html_file.close()
    
    if(total_files_to_scrape!=files_scraped):
        print("Some files failed to scrape\n")
        text="Some files failed to scrape\n" + str(total_files_to_scrape) + "!="+ str(files_scraped) + "\n"
        SaveLogFile(log_file_name, text)
        
    SaveLogFile(log_file_name, "saving log files==========================\n")
    CloseLogFile(log_file_name)
    os.chdir('..')
        
    return

#### Now we call the Extract10K function to scrape 10-K sec filings

In [9]:
# To scrape 10-Ks, call the Extract10K method
# Define parameters for Extract10K method
browse_url_10k = 'https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=%s&type=10-K'
filing_url_10k = 'http://www.sec.gov/Archives/edgar/data/%s/%s-index.html'
doc_url_10k = 'http://www.sec.gov/Archives/edgar/data/%s/%s/%s'

# moving the directory, where scraped 10-K filings are
os.chdir(scraped10k_path)
log_file = 'log.txt'

for cik in tqdm(ciks_tick_df['cik']):
    Extract10K(browse_url_base=browse_url_10k, 
          filing_url_base=filing_url_10k, 
          doc_url_base=doc_url_10k, 
          cik=cik,
          log_file_name=log_file)

  0%|          | 0/2 [00:00<?, ?it/s]

Scraping CIK  0000066740


 50%|█████     | 1/2 [00:00<00:00,  1.42it/s]

Scraping CIK  0000001800


100%|██████████| 2/2 [00:01<00:00,  1.47it/s]


#### Now we call the Extract10Q function to scrape 10-K sec filings

In [10]:
# To scrape 10-Ks, call the Extract10Q method
# Define parameters for Extract10Q method
browse_url_10q = 'https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=%s&type=10-Q&count=1000'
filing_url_10q = 'http://www.sec.gov/Archives/edgar/data/%s/%s-index.html'
doc_url_10q = 'http://www.sec.gov/Archives/edgar/data/%s/%s/%s'

# Set correct directory (fill this out yourself!)
os.chdir(scraped10q_path)
log_file_name = 'log.txt'

# moving the directory, where scraped 10-K filings are
for cik in tqdm(ciks_tick_df['cik']):
    Extract10Q(browse_url_base=browse_url_10q, 
          filing_url_base=filing_url_10q, 
          doc_url_base=doc_url_10q, 
          cik=cik,
          log_file_name=log_file_name)

  0%|          | 0/2 [00:00<?, ?it/s]

Scraping CIK  0000066740


 50%|█████     | 1/2 [00:00<00:00,  1.36it/s]

Scraping CIK  0000001800


100%|██████████| 2/2 [00:01<00:00,  1.42it/s]


## 3. Parsing the HTML files in text files

#### remove numerical tables

In [14]:
def RemoveNumberTables(soup):
    
    def GetNumPercentage(tablestring):
        if len(tablestring)>0.0:
            numbers = sum([char.isdigit() for char in tablestring])
            length = len(tablestring)
            return numbers/length
        else:
            return 1
        
    [x.extract() for x in soup.find_all('table') if GetNumPercentage(x.get_text())>0.15]
    
    return soup

#### remove html tags from files

In [15]:
def RemoveHTMLTags(soup):
    
    text = soup.get_text()
    text = text.replace('\n', ' ')
    text = unicodedata.normalize('NFKD', text)
    
    return text

#### parse html data into textual data

In [13]:
def ParseHTMLToText(cik):
    try: 
        os.chdir(cik)
  
    except FileNotFoundError:
        print("Could not find directory for CIK", cik)
        return
        
    print("Parsing CIK %s..." % cik)
    #setting the flag to check if a flie already parsed or not
    parsed = False 
    
    # Creating a new directory 'clean_data' within the CIK directory
    # and storing text data of filings
    if os.path.exists('rawtext'):
        os.makedirs('clean_data')
        os.rename("rawtext", "clean_data")
    elif not os.path.exists('clean_data'):
        os.makedirs('clean_data')
      
    file_list = [fname for fname in os.listdir() if not (fname.startswith('.') | os.path.isdir(fname) | fname.startswith('log'))]
 
    for filename in file_list:
        new_filename = filename.replace('.html', '.txt')
        text_file_list = os.listdir('clean_data')
        if new_filename in text_file_list:
            continue
               
        #Cleaning the file by removing numerical tables, tags etc.
        with open(filename, 'r') as file:
            parsed = True
            soup = bs.BeautifulSoup(file.read(), "lxml")
            soup = RemoveNumberTables(soup)
            text = RemoveHTMLTags(soup)
            hex_handle = re.sub(r'[^\x00-\x7f]',' ', text)
            with open('clean_data/'+new_filename, 'w') as newfile:
                newfile.write(hex_handle)
    
    # Once all the files parsed, log them
    if parsed==False:
        print("Already parsed CIK", cik)
    
    os.chdir('..')
    return

#### We can now apply this function to each of our 10-K and 10-Q files.

In [16]:
# Parsing the 10-K files
os.chdir(scraped10k_path)
for cik in tqdm(ciks_tick_df['cik']):
    ParseHTMLToText(cik)

100%|██████████| 2/2 [00:00<00:00, 242.37it/s]

Parsing CIK 0000066740...
Already parsed CIK 0000066740
Parsing CIK 0000001800...
Already parsed CIK 0000001800


In [17]:
# Parsing the 10-Q files
os.chdir(scraped10q_path)
for cik in tqdm(ciks_tick_df['cik']):
    ParseHTMLToText(cik)

100%|██████████| 2/2 [00:00<00:00, 174.81it/s]

Parsing CIK 0000066740...
Already parsed CIK 0000066740
Parsing CIK 0000001800...
Already parsed CIK 0000001800
